In [1]:
# Task 1: Setup the LLM API Connection
import requests
from google.colab import userdata

api_key = userdata.get('OPENROUTER_API_KEY')

def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "HTTP-Referer": "https://colab.research.google.com/",
        "Content-Type": "application/json"
    }
    data = {
        "model": "meta-llama/llama-3.1-8b-instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    response = requests.post(url, headers=headers, json=data)

    if response.status_code == 200:
        return response.json()['choices'][0]['message']['content'].strip()
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

print(call_llm("You are a helpful assistant.", "Say hello"))

Hello! How can I assist you today?


In [2]:
# Task 2: Updated Prompt Design
system_prompt = """
You are a credit scoring assistant.
Return the output STRICTLY as a valid JSON object with these 5 fields:
{"risk_tier": "low|medium|high", "confidence": "low|medium|high", "recommended_action": "string", "primary_signal": "string", "flag_for_review": true|false}

RULES:
1. Return ONLY the JSON object.
2. DO NOT include any introductory or concluding text, reasoning, or explanations.
3. DO NOT use markdown formatting (like ```json).
4. If you add even a single extra word outside the JSON, the system will fail.
"""
response = call_llm(system_prompt, '{"income": 20000, "debt": 40000, "credit_score": 500}', temperature=0.0)
print(response)

{"risk_tier": "high", "confidence": "medium", "recommended_action": "Pay down debt aggressively", "primary_signal": "High debt-to-income ratio", "flag_for_review": true}


In [3]:
# Task 2: Temperature A/B Comparison

user_prompt = '{"income": 20000, "debt": 40000, "credit_score": 500}'

# Strict system prompt to ensure ONLY JSON output

strict_system_prompt = system_prompt + "\nIMPORTANT: Return ONLY the JSON object. Do not include any reasoning or extra text."

print("--- Output at Temperature 0.0 ---")
print(call_llm(strict_system_prompt, user_prompt, temperature=0.0))

print("\n--- Output at Temperature 0.7 ---")
print(call_llm(strict_system_prompt, user_prompt, temperature=0.7))

--- Output at Temperature 0.0 ---
{"risk_tier": "high", "confidence": "medium", "recommended_action": "Pay down debt and improve credit score", "primary_signal": "high debt", "flag_for_review": true}

--- Output at Temperature 0.7 ---
{"risk_tier": "high", "confidence": "medium", "recommended_action": "Decrease debt to improve credit score", "primary_signal": "High debt relative to income", "flag_for_review": false}


In [4]:
# Task 3: Tabular Record Batch Scoring
import pandas as pd
import json
from jsonschema import validate, ValidationError

schema = {
    "type": "object",
    "properties": {
        "risk_tier": {"type": "string"},
        "confidence": {"type": "string"},
        "recommended_action": {"type": "string"},
        "primary_signal": {"type": "string"},
        "flag_for_review": {"type": "boolean"}
    },
    "required": ["risk_tier", "confidence", "recommended_action", "primary_signal", "flag_for_review"]
}
df = pd.read_csv('cleaned_data.csv')
df =df.head(5)

records = df.to_json(orient='records', lines=True).split('\n')

for record in records:
    if not record.strip(): continue

    response = call_llm(system_prompt, record, temperature=0.0)

    if not response or response.strip() == "":
        print("Status: Skipped (Empty response from LLM)")
        continue

    try:

        import re
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            data = json.loads(json_match.group())
            validate(instance=data, schema=schema)
            print("Status: Pass")
        else:
            print("Status: Fail - No JSON found in response")
    except Exception as e:
        print(f"Status: Fail. Error: {e}. Response was: {response}")

Status: Pass
Status: Pass
Status: Pass
Status: Pass
Status: Pass


In [5]:
## Task 4: Guardrails

import re

def has_pii(text):
    # Email and Phone pattern
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'

    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))

def safe_call_llm(system_prompt, user_input, temperature=0.0):
    if has_pii(user_input):
        print("Input blocked: PII detected.")
        return None
    else:
        return call_llm(system_prompt, user_input, temperature)

# Testing the Guardrail
test_input_1 = "Contact me at shivani@email.com"
test_input_2 = '{"income": 20000, "debt": 40000, "credit_score": 500}'

print("Test 1 Result:")
safe_call_llm("You are a helpful assistant.", test_input_1)

print("\nTest 2 Result:")
safe_call_llm(strict_system_prompt, test_input_2)

Test 1 Result:
Input blocked: PII detected.

Test 2 Result:


'{"risk_tier": "high", "confidence": "high", "recommended_action": "Improve credit score and reduce debt", "primary_signal": "high debt", "flag_for_review": false}'

In [7]:
## Task 5: End-to-End Demonstration

import pandas as pd
import json
import re
from jsonschema import validate

# Load dataset
df_demo = pd.read_csv('cleaned_data.csv').head(3)
records = df_demo.to_json(orient='records', lines=True).split('\n')

results = []

# Pipeline execution
for record in records:
    if not record.strip():
        continue

    # PII Guardrail
    if has_pii(record):
        validation_status = "Blocked (PII Detected)"
        llm_output = "N/A"
    else:
        # LLM Call
        response = call_llm(strict_system_prompt, record, temperature=0.0)

        # Structured Output Handling & Validation
        try:
            json_match = re.search(r'\{.*\}', response, re.DOTALL)
            if json_match:
                data = json.loads(json_match.group())
                validate(instance=data, schema=schema)
                validation_status = "Pass"
                llm_output = str(data)
            else:
                validation_status = "Fail (No JSON)"
                llm_output = response
        except Exception as e:
            validation_status = f"Fail: {e}"
            llm_output = response

    results.append({"Input": record, "LLM Output": llm_output, "Validation Status": validation_status})

# Display Results
results_df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', 50)
print(results_df)

                                               Input  \
0  {"Hours_Studied":23,"Attendance":84,"Parental_...   
1  {"Hours_Studied":19,"Attendance":64,"Parental_...   
2  {"Hours_Studied":24,"Attendance":98,"Parental_...   

                                          LLM Output Validation Status  
0  {'risk_tier': 'high', 'confidence': 'medium', ...              Pass  
1  {'risk_tier': 'high', 'confidence': 'medium', ...              Pass  
2  {'risk_tier': 'low', 'confidence': 'high', 're...              Pass  
